### TMA settle test
Author: Nacho Sevilla - May 2026

This notebook plots the distribution of settle times defined as the first time the derivative of the (actual - demand) positions changes sign after crossing zero value.



In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from astropy.time import Time

from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState
from lsst.summit.utils.tmaUtils import plotEvent, getAzimuthElevationDataForEvent


In [ ]:
def analyze_settle_time(
    client,
    event,
    pre_padding=10,
    post_padding=10,
    make_plot=False,
):
    """Calculate settle time for an event.

        With optional padding before and after the event. 

        Parameters
        ----------
        client : `EfdClient`, for the session
        event: `TMAEvent`, event to be analyzed
        pre_padding: `int`, seconds before start of event to retrieve for analysis
        post_padding: `int`, seconds after end of event to retrieve for analysis
        make_plot: `bool`, show a position and residuals plot for the event

        Returns
        -------
        settle_time: `float`, settle time defined as the time between the first overshoot and time that TMA starts TRACKING
    """

    df_mtmount_azi,df_mtmount_ele = getAzimuthElevationDataForEvent(client,event,pre_padding,post_padding)
    df_act_azi = df_mtmount_azi["actualPosition"] 
    df_dem_azi = df_mtmount_azi["demandPosition"]
    df_act_ele = df_mtmount_ele["actualPosition"] 
    df_dem_ele = df_mtmount_ele["demandPosition"]
    azi_residuals = 3600 * (df_act_azi - df_dem_azi) # in arcseconds
    ele_residuals = 3600 * (df_act_ele - df_dem_ele) # in arcseconds
    t_end = pd.to_datetime(event.end.isot)
    t_end = t_end.tz_localize("UTC")

    velocity_threshold = 0.05
    slew_mask_azi = np.abs(df_mtmount_azi["demandVelocity"]) > velocity_threshold
    slew_mask_ele = np.abs(df_mtmount_ele["demandVelocity"]) > velocity_threshold
    if slew_mask_azi.sum() > slew_mask_ele.sum():
        slew_mask = slew_mask_azi
    else:
        slew_mask = slew_mask_ele
    slew_start_idx = np.argmax(slew_mask)
    azi_overshoot = pd.to_datetime(determine_overshoot_time(azi_residuals.iloc[slew_start_idx:]))
    ele_overshoot = pd.to_datetime(determine_overshoot_time(ele_residuals.iloc[slew_start_idx:]))
    
    settle_time_azi = (t_end - azi_overshoot).total_seconds()
    settle_time_ele = (t_end - ele_overshoot).total_seconds()

    if make_plot:
        fig, axes = plt.subplots(
            4,
            1,
            sharex=True,
            figsize=(12, 10),
            gridspec_kw={"height_ratios": [3, 1, 3, 1]}
        )
        ax1, ax2, ax3, ax4 = axes
    
        ax1.plot(df_act_azi, color="red", lw="0.5", label="Actual")
        ax1.plot(df_dem_azi, color="blue", lw="0.5", label="Demand")
        ax1.axvline(t_end, lw="1.25", c="k", ls="dashed", label="TRACKING")
        ax1.axvline(azi_overshoot, lw="1.25", c="green", ls="dashed", label="Overshoot")
        ax1.set_ylabel(f"Azimuth position")
        #ax1.set_ylim(-0.01,0.01)
        ax1.legend()
        ax1.grid(True)
    
        ax2.plot(azi_residuals)
        ax2.axvline(t_end, lw="1.25", c="k", ls="dashed", label="Event end")
        ax2.axhline(-0.01, lw="1.25", c="k", ls="dashed")
        ax2.axhline(0.01, lw="1.25", c="k", ls="dashed")
        ax2.set_ylabel("Residual")
        ax2.set_ylim(-0.1,0.1)
        ax2.grid(True)
    
        ax3.plot(df_act_ele, color="red", lw="0.5", label="Actual")
        ax3.plot(df_dem_ele, color="blue", lw="0.5", label="Demand")
        ax3.axvline(t_end, lw="1.25", c="k", ls="dashed", label="TRACKING")
        ax3.axvline(ele_overshoot, lw="1.25", c="green", ls="dashed", label="Overshoot")
        ax3.set_ylabel(f"Elevation position")
        #ax3.set_ylim(-0.01,0.01)
        ax3.legend()
        ax3.grid(True)
    
        ax4.plot(ele_residuals)
        ax4.axvline(t_end, lw="1.25", c="k", ls="dashed", label="Event end")
        ax4.axhline(-0.01, lw="1.25", c="k", ls="dashed")
        ax4.axhline(0.01, lw="1.25", c="k", ls="dashed")
        ax4.set_ylabel("Residual")
        ax4.set_ylim(-0.1,0.1)
        ax4.grid(True)
    
        ax4.set_xlabel("UTC")
        fig.autofmt_xdate()
        fig.subplots_adjust(hspace=1)
        fig.tight_layout()

    # we select the most conservative settle_time
    # note that a negative settle time means that the telescope has reached TRACKING conditions without previous overshoot to settle
    settle_time = max(settle_time_azi, settle_time_ele) 

    return settle_time
 

In [ ]:
def determine_overshoot_time(residual):
    #
    """Calculate overshoot time as the first point where the derivative of the residual changes sign after residual = 0 
        Parameters
        ----------
        residual : `pandas Series`, with the actual - demand position for a given axis

        Returns
        -------
        overshoot_time: `pandas Timestamp`, in UTC, time where overshoot happens according to the definition above
    """
    # numerical derivative
    dr = np.gradient(residual.values)
    
    # sign of derivative
    sign = np.sign(dr)
    
    # turning points
    turning = np.where(np.diff(sign))[0]
    
    # first turning point after zero crossing
    zc = np.where(np.diff(np.sign(residual.values)))[0]

    overshoot_time = residual.index[-1] #set default values of overshoot time to the end of the window, so it returns a value in case no overshoot detected
    overshoot_value = residual.iloc[-1]
    
    if len(zc) > 0:
        first_zc = zc[0]
    
        candidates = turning[turning > first_zc]
    
        if len(candidates) > 0:
            idx = candidates[0]
    
            overshoot_time = residual.index[idx]
            overshoot_value = residual.iloc[idx]
    
            #print("Overshoot at:", overshoot_time)
            #print("Residual:", overshoot_value)

    return overshoot_time

In [ ]:
# Create an EFD client instance
client = makeEfdClient()
eventMaker = TMAEventMaker()

day_obs = 20260517

In [ ]:
# Select data from a given date
eventMaker = TMAEventMaker()
events = eventMaker.getEvents(day_obs)

# Get lists of slew and track events
slews = [e for e in events if e.type == TMAState.SLEWING]
tracks = [e for e in events if e.type == TMAState.TRACKING]
print(f"Found {len(slews)} slews and {len(tracks)} tracks")

In [ ]:
#plotEvent(client,events[746],fig=None,prePadding=10,postPadding=10,doFilterResiduals=True)

In [ ]:
analyze_settle_time(client,events[231],pre_padding=1,post_padding=10, make_plot=True)

Go through all events in day_obs and make a histogram of the settle_times

In [ ]:
settle_time = []
for ev, event in enumerate(events):
    if ev % 100 == 0:
        print(f'{ev}/{len(events)}')
    # limit to slews that end in tracking, and slews of less than 8 s (such as those from regular FBS)
    if (event.type == TMAState.SLEWING) and (event.endReason == TMAState.TRACKING and event.duration > 2 and event.duration < 8): 
        st = analyze_settle_time(client,event,pre_padding=1,post_padding=10, make_plot=False)
        settle_time.append(st)
        if (st > 3):
            print(ev, st)
plt.hist(settle_time,bins=30)
plt.xlabel('TMA settling time')
plt.suptitle(f'day_obs {day_obs}')